# Set up RelationalAI (PyRel) in a Snowflake Notebook

This notebook checks your environment, installs PyRel, connects through the notebook's own session, and runs a first query on the supply-chain tables. Afterwards, continue with `02_supplier_risk_to_plan.ipynb`.

> **Generated with AI.** The code and text in this notebook were generated with generative AI (Cortex Code), not handwritten, then run and checked. Review before reusing.

**Prerequisites** (see `scripts/`):
1. The RelationalAI Native App is installed and activated in your account.
2. You have run `load_data.sql`, which creates the role `RAI_QS_ROLE`, the warehouse `RAI_QS_WH` and the tables in `RAI_QUICKSTART.SUPPLY_CHAIN`.
3. You have run `admin_setup.sql`, which creates the external access integration `RAI_NOTEBOOK_ACCESS` for PyPI and RelationalAI query results.
4. This notebook runs on **Container Runtime** (compute pool `SYSTEM_COMPUTE_POOL_CPU`), as role `RAI_QS_ROLE`, with `RAI_NOTEBOOK_ACCESS` enabled under *Notebook settings → External access*.

In [ ]:
# 1. Pre-flight: Python version and Snowflake context (prints no secrets).
import sys
from snowflake.snowpark.context import get_active_session

session = get_active_session()
role, wh, db = session.sql("SELECT CURRENT_ROLE(), CURRENT_WAREHOUSE(), CURRENT_DATABASE()").collect()[0]
checks = {
    "Python >= 3.10": sys.version_info[:2] >= (3, 10),
    "RelationalAI app present": len(session.sql("SHOW APPLICATIONS LIKE 'RELATIONALAI'").collect()) == 1,
    "Sample tables loaded": session.sql(
        "SELECT COUNT(*) FROM RAI_QUICKSTART.INFORMATION_SCHEMA.TABLES WHERE TABLE_SCHEMA = 'SUPPLY_CHAIN' AND TABLE_TYPE = 'BASE TABLE'"
    ).collect()[0][0] >= 10,
}
print(f"Python {sys.version.split()[0]} | role {role} | warehouse {wh} | database {db}")
for name, ok in checks.items():
    print(("OK  " if ok else "FAIL"), name)

## 2. Install PyRel, then restart the kernel

`relationalai` is not in the Snowflake Anaconda channel, so it's installed from PyPI through `RAI_NOTEBOOK_ACCESS`. pip reports dependency conflicts for `mlruntimes-service` and `opentelemetry-proto`; you can ignore them for this guide.

**Restart the kernel after the install.** The install upgrades `protobuf` from 4.25 to 5.x, but the running kernel keeps the old version loaded. Without a restart, the first query fails with `ImportError: cannot import name 'runtime_version' from 'google.protobuf'`.

In [ ]:
%pip install -q "relationalai>=1.30.1,<2.0" networkx
# Now restart the kernel, then continue with the next cell (don't rerun this one).

In [ ]:
# 3. Check the install and connect. In a notebook, PyRel reuses the active session:
#    no raiconfig.yaml, no password, no token.
import importlib.metadata as md
from relationalai.config import create_config

print("relationalai", md.version("relationalai"), "| protobuf", md.version("protobuf"))
cfg = create_config()
print(type(cfg).__name__)   # expected: ConfigFromActiveSession

In [ ]:
# 4. A first, tiny model and query: which tier-1 suppliers supply the temperature sensor P-141?
#    The first query takes several minutes: RelationalAI starts an engine and indexes the tables.
from relationalai.semantics import Model, String

model = Model("SetupCheck", config=cfg)
Supplier = model.Concept("Supplier", identify_by={"id": String})
Supplier.region = model.Property(f"{Supplier} is in region {String:region}")
Supplier.supplies = model.Relationship(f"{Supplier} supplies part {String:part_id}")

suppliers = model.Table("RAI_QUICKSTART.SUPPLY_CHAIN.SUPPLIERS")
supplier_parts = model.Table("RAI_QUICKSTART.SUPPLY_CHAIN.SUPPLIER_PARTS")
model.define(s := Supplier.new(id=suppliers.SUPPLIER_ID), s.region(suppliers.REGION))
model.define(Supplier.lookup(id=supplier_parts.SUPPLIER_ID).supplies(supplier_parts.PART_ID))

result = model.where(Supplier.supplies("P-141")).select(
    Supplier.id.alias("supplier"), Supplier.region.alias("region")).to_df().sort_values("supplier")
print(result)
assert list(result["supplier"]) == ["S-T1-02", "S-T1-05"]
print("Setup works. Continue with 02_supply_chain_better_together.ipynb")